# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant[full] --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic metadata (access as attributes, not dict)
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review available record sets and their fields, referenced by their `@id`.

In [ ]:
# List all available record sets in the dataset (by @id and name, if available)
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    print('Available record sets:')
    for record_set in metadata.record_sets:
        info = f"@id: {record_set.id}"
        if hasattr(record_set, 'name') and record_set.name:
            info += f", name: {record_set.name}"
        print(info)
        if hasattr(record_set, 'fields'):
            for field in record_set.fields:
                field_info = f"    Field @id: {field.id}"
                if hasattr(field, 'name') and field.name:
                    field_info += f", name: {field.name}"
                print(field_info)
else:
    print('No record sets defined in dataset metadata.')

## 3. Data Extraction
Load data from all available record sets into DataFrames for further analysis and exploration. All fields and columns should be referenced by their `@id`.

In [ ]:
# Extract all record sets @id to load the data
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    record_set_ids = [rs.id for rs in metadata.record_sets]
else:
    record_set_ids = []

print('Record Set @ids to load:', record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from record set @id: {record_set_id}")
        else:
            print(f"No records found for record set @id: {record_set_id}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# For demonstration, pick the first available record set (if any)
if dataframes:
    first_rs_id = next(iter(dataframes.keys()))
    df = dataframes[first_rs_id]
    print(f"\nColumns for record set @id '{first_rs_id}':\n", df.columns.tolist())
    display(df.head())
else:
    print('No dataframes loaded. Please check the record sets or dataset definition.')

## 4. Exploratory Data Analysis (EDA)
Apply example data processing: filter records based on a numeric field, normalize values, and group by categorical field. All references use the `@id` used in the data extraction step above.

In [ ]:
# --- EDA: Customize below as relevant to dataset columns ---
if dataframes:
    # Use the first DataFrame loaded
    record_set_id = first_rs_id
    df = dataframes[record_set_id]
    
    print(f"Performing EDA on record set @id: {record_set_id}\n")
    
    # Show data types for quick variable selection
    print('Column types:')
    print(df.dtypes)
    
    # Identify numeric and categorical fields
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"\nUsing numeric field (by @id): {numeric_field_id}")

        # Example filter: values above median
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold].copy()

        print(f"Filtered records with {numeric_field_id} > {threshold} (median): {len(filtered_df)} rows")
        display(filtered_df.head())

        # Normalize this field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized field '{numeric_field_id}':")
        display(filtered_df[[numeric_field_id, norm_col]].head())
    else:
        print('No numeric fields found for filtering and normalization.')

    # Example grouping by a categorical field (if available)
    if cat_cols:
        group_field_id = cat_cols[0]
        print(f"\nGrouping by categorical field (by @id): {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
        print(grouped_df.head())
    else:
        print('No categorical fields found for grouping.')
else:
    print('No data available for EDA. Please ensure data is loaded.')

## 5. Visualization
Visualize the distribution of a numeric field and explore relationships between two fields using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure we have results from prior EDA
if dataframes and numeric_cols:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' (@id)")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If a categorical field exists, show barplot of mean by category
    if cat_cols:
        plt.figure(figsize=(8,4))
        # Only take categories with sufficient data (>5 samples)
        cat_counts = df[cat_cols[0]].value_counts()
        valid_cats = cat_counts[cat_counts > 5].index
        sns.barplot(
            x=cat_cols[0], y=numeric_field_id,
            data=df[df[cat_cols[0]].isin(valid_cats)],
            ci='sd', estimator='mean'
        )
        plt.title(f"Mean of '{numeric_field_id}' by '{cat_cols[0]}' (@id)")
        plt.xlabel(cat_cols[0])
        plt.ylabel(f"mean of {numeric_field_id}")
        plt.xticks(rotation=30, ha='right')
        plt.show()
else:
    print('Visualization skipped: No suitable numeric and categorical data available.')

## 6. Conclusion
This notebook demonstrated the loading, overview, and exploratory analysis workflow on the FAIR² dataset using the `mlcroissant` library, referencing all record sets, fields, and columns by their `@id`. For more advanced analyses, use the full Croissant metadata to optimize ingestion and leverage relationships between multiple record sets if available.

<i>This notebook and the FAIR² dataset empower transparent, reproducible research and responsible data science in rangeland management and social-ecological systems.</i>